# CNN CPU vs GPU Analysis
Averages triplicates (rep 0, 1, 2) and computes standard deviation across runs.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('/Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/CNN')
OUTPUT_DIR = Path('/Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/CNN')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Columns that identify a unique experimental condition (grouping keys)
GROUP_COLS = ['model_type', 'dataset', 'model_name', 'pruning_method',
              'stored_precision', 'architecture', 'device']

# Numeric measurement columns to average and compute std over triplicates
METRIC_COLS = [
    'mean_latency_ms', 'std_latency_ms', 'median_latency_ms',
    'p25_latency_ms', 'p75_latency_ms', 'p90_latency_ms',
    'min_latency_ms', 'max_latency_ms', 'throughput_imgs_per_s',
    'mean_energy_kwh_per_image', 'mean_cpu_power_w', 'mean_gpu_power_w',
    'mean_ram_power_w'
]

# Columns that are constant across reps -- just take first value
CONST_COLS = ['num_params', 'model_size_mb', 'num_images']

datasets = ['bloodmnist', 'chestmnist', 'dermamnist', 'pathmnist']
all_dfs = []

for dataset in datasets:
    csv_path = DATA_DIR / f'{dataset}_results.csv'
    df = pd.read_csv(csv_path)
    all_dfs.append(df)
    print(f'Loaded {csv_path.name}: {len(df)} rows')

combined = pd.concat(all_dfs, ignore_index=True)
print(f'\nCombined: {len(combined)} rows across {combined["dataset"].nunique()} datasets')
print(f'Reps per condition check -- unique rep values: {sorted(combined["rep"].unique())}')

# Compute mean and std across triplicates
agg_mean = combined.groupby(GROUP_COLS)[METRIC_COLS].mean().reset_index()
agg_std  = combined.groupby(GROUP_COLS)[METRIC_COLS].std(ddof=1).reset_index()

# Rename: original name -> _mean and _std suffixes
agg_mean = agg_mean.rename(columns={c: f'{c}_mean' for c in METRIC_COLS})
agg_std  = agg_std.rename(columns={c: f'{c}_std'  for c in METRIC_COLS})

# Constant columns: take first rep value
const_vals = combined.groupby(GROUP_COLS)[CONST_COLS].first().reset_index()

# Merge everything
result = agg_mean.merge(agg_std, on=GROUP_COLS).merge(const_vals, on=GROUP_COLS)

# Reorder: group cols | const cols | interleaved mean/std pairs
interleaved = [col for metric in METRIC_COLS for col in (f'{metric}_mean', f'{metric}_std')]
result = result[GROUP_COLS + CONST_COLS + interleaved]

print(f'Output shape: {result.shape}')
result.head()

# Save
out_path = OUTPUT_DIR / 'CNN_averaged.csv'
result.to_csv(out_path, index=False)
print(f'Saved to {out_path}')
result

Loaded bloodmnist_results.csv: 48 rows
Loaded chestmnist_results.csv: 48 rows
Loaded dermamnist_results.csv: 48 rows
Loaded pathmnist_results.csv: 48 rows

Combined: 192 rows across 4 datasets
Reps per condition check -- unique rep values: [0, 1, 2]


In [6]:
"""
cnn_visualize.py

Generates charts and summary tables from CNN_averaged.csv.

Usage:
    python cnn_visualize.py \
        --input  /path/to/CNN_averaged.csv \
        --output /path/to/output_dir

Chart encoding:
    x-axis   : compression method  (baseline | structured_pruning | hybrid | knowledge_distillation)
    color    : device               (CPU = green, GPU = red)
    marker   : stored precision     (fp32 = triangle ▲, fp16 = circle ●)
    panels   : one column per dataset (bloodmnist | chestmnist | dermamnist | pathmnist)
"""

from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

# Map raw pruning_method → display label
# quantization is remapped to "baseline" in load_data so it appears as the
# fp16 counterpart of the fp32 baseline — same x-position, distinguished by marker.
METHOD_LABELS = {
    "baseline":         "Baseline /\nQuantization",
    "hybrid_pruning":   "Hybrid",
    "regional_pruning": "Structured\nPruning",
    "slim_kd":          "Knowledge\nDistillation",
}

METHOD_ORDER = ["baseline", "regional_pruning", "hybrid_pruning", "slim_kd"]

DATASETS = ["bloodmnist", "chestmnist", "dermamnist", "pathmnist"]
DATASET_LABELS = {
    "bloodmnist": "BloodMNIST",
    "chestmnist": "ChestMNIST",
    "dermamnist": "DermaMNIST",
    "pathmnist":  "PathMNIST",
}

# Visual encoding
DEVICE_COLORS = {"cpu": "#2e7d32", "cuda": "#c62828"}   # green / red
PRECISION_MARKERS = {"fp32": "^", "fp16": "o"}           # triangle / circle
PRECISION_SIZES = {"fp32": 80, "fp16": 80}

# Global style
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "figure.dpi": 150,
})

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def load_data(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    # Normalise column names: strip whitespace
    df.columns = df.columns.str.strip()
    # Keep only CNN rows (script is CNN-focused, but be safe)
    if "model_type" in df.columns:
        df = df[df["model_type"] == "CNN"].copy()

    # Remap quantization → baseline so it sits at the same x-position as the
    # fp32 baseline, distinguished only by the fp16 circle marker.
    # This mirrors how regional_pruning_fp16 pairs with regional_pruning_fp32.
    df["pruning_method"] = df["pruning_method"].replace("quantization", "baseline")

    # Add display label for method
    df["method_label"] = df["pruning_method"].map(METHOD_LABELS).fillna(df["pruning_method"])
    df["method_order"] = df["pruning_method"].map(
        {m: i for i, m in enumerate(METHOD_ORDER)}
    ).fillna(99)
    return df


def jitter(n: int, width: float = 0.08) -> np.ndarray:
    """Small horizontal jitter so overlapping points are visible."""
    rng = np.random.default_rng(42)
    return rng.uniform(-width, width, n)


def _x_positions(methods: list, method_order: list) -> dict:
    """Map method string → x position (0, 1, 2 …)."""
    ordered = [m for m in method_order if m in methods]
    return {m: i for i, m in enumerate(ordered)}


def _legend_handles():
    """Build shared legend handles for device colour + precision marker."""
    handles = []
    for device, color in DEVICE_COLORS.items():
        patch = mpatches.Patch(color=color, label=f"Device: {device.upper()}")
        handles.append(patch)
    handles.append(mlines.Line2D([], [], color="gray", marker="^", linestyle="None",
                                  markersize=8, label="FP32"))
    handles.append(mlines.Line2D([], [], color="gray", marker="o", linestyle="None",
                                  markersize=8, label="FP16"))
    return handles


# ---------------------------------------------------------------------------
# Figure 1: Mean latency (ms) — one panel per dataset
# ---------------------------------------------------------------------------

def plot_latency(df: pd.DataFrame, output_dir: Path):
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(18, 5), sharey=False)
    fig.suptitle("Single-Image Inference Latency by Compression Method",
                 fontsize=14, fontweight="bold", y=1.02)

    for ax, dataset in zip(axes, DATASETS):
        sub = df[df["dataset"] == dataset].copy()
        if sub.empty:
            ax.set_visible(False)
            continue

        methods_present = [m for m in METHOD_ORDER if m in sub["pruning_method"].values]
        x_pos = _x_positions(methods_present, METHOD_ORDER)

        for _, row in sub.iterrows():
            m = row["pruning_method"]
            if m not in x_pos:
                continue
            x = x_pos[m]
            device = row["device"]
            precision = row["stored_precision"]
            color = DEVICE_COLORS.get(device, "gray")
            marker = PRECISION_MARKERS.get(precision, "s")
            size = PRECISION_SIZES.get(precision, 80)

            y = row["mean_latency_ms_mean"]
            yerr = row.get("mean_latency_ms_std", 0)

            ax.errorbar(
                x, y, yerr=yerr,
                fmt="none", color=color, alpha=0.6, capsize=4, linewidth=1.2,
            )
            ax.scatter(x, y, color=color, marker=marker, s=size,
                       zorder=5, edgecolors="white", linewidths=0.5)

        ax.set_xticks(list(range(len(methods_present))))
        ax.set_xticklabels(
            [METHOD_LABELS.get(m, m) for m in methods_present],
            fontsize=8, rotation=20, ha="right"
        )
        ax.set_title(DATASET_LABELS.get(dataset, dataset), fontsize=10, fontweight="bold")
        ax.set_ylabel("Mean Latency (ms)" if ax == axes[0] else "")
        ax.set_xlabel("")

    fig.legend(handles=_legend_handles(), loc="lower center", ncol=4,
               bbox_to_anchor=(0.5, -0.12), frameon=False, fontsize=9)
    plt.tight_layout()
    out = output_dir / "fig1_latency_by_method.png"
    fig.savefig(out, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")


# ---------------------------------------------------------------------------
# Figure 2: Throughput (img/s)
# ---------------------------------------------------------------------------

def plot_throughput(df: pd.DataFrame, output_dir: Path):
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(18, 5), sharey=False)
    fig.suptitle("Single-Image Throughput by Compression Method",
                 fontsize=14, fontweight="bold", y=1.02)

    for ax, dataset in zip(axes, DATASETS):
        sub = df[df["dataset"] == dataset].copy()
        if sub.empty:
            ax.set_visible(False)
            continue

        methods_present = [m for m in METHOD_ORDER if m in sub["pruning_method"].values]
        x_pos = _x_positions(methods_present, METHOD_ORDER)

        for _, row in sub.iterrows():
            m = row["pruning_method"]
            if m not in x_pos:
                continue
            x = x_pos[m]
            color = DEVICE_COLORS.get(row["device"], "gray")
            marker = PRECISION_MARKERS.get(row["stored_precision"], "s")
            size = PRECISION_SIZES.get(row["stored_precision"], 80)

            y = row["throughput_imgs_per_s_mean"]
            yerr = row.get("throughput_imgs_per_s_std", 0)

            ax.errorbar(x, y, yerr=yerr, fmt="none", color=color,
                        alpha=0.6, capsize=4, linewidth=1.2)
            ax.scatter(x, y, color=color, marker=marker, s=size,
                       zorder=5, edgecolors="white", linewidths=0.5)

        ax.set_xticks(list(range(len(methods_present))))
        ax.set_xticklabels(
            [METHOD_LABELS.get(m, m) for m in methods_present],
            fontsize=8, rotation=20, ha="right"
        )
        ax.set_title(DATASET_LABELS.get(dataset, dataset), fontsize=10, fontweight="bold")
        ax.set_ylabel("Throughput (img/s)" if ax == axes[0] else "")

    fig.legend(handles=_legend_handles(), loc="lower center", ncol=4,
               bbox_to_anchor=(0.5, -0.12), frameon=False, fontsize=9)
    plt.tight_layout()
    out = output_dir / "fig2_throughput_by_method.png"
    fig.savefig(out, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")


# ---------------------------------------------------------------------------
# Figure 3: Energy per image (kWh)
# ---------------------------------------------------------------------------

def plot_energy(df: pd.DataFrame, output_dir: Path):
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(18, 5), sharey=False)
    fig.suptitle("Mean Energy per Single Inference by Compression Method",
                 fontsize=14, fontweight="bold", y=1.02)

    for ax, dataset in zip(axes, DATASETS):
        sub = df[df["dataset"] == dataset].copy()
        if sub.empty:
            ax.set_visible(False)
            continue

        methods_present = [m for m in METHOD_ORDER if m in sub["pruning_method"].values]
        x_pos = _x_positions(methods_present, METHOD_ORDER)

        for _, row in sub.iterrows():
            m = row["pruning_method"]
            if m not in x_pos:
                continue
            x = x_pos[m]
            color = DEVICE_COLORS.get(row["device"], "gray")
            marker = PRECISION_MARKERS.get(row["stored_precision"], "s")
            size = PRECISION_SIZES.get(row["stored_precision"], 80)

            # energy (µWh) = (latency_ms / 1000) * power_W / 3600 * 1e6
            lat   = row["mean_latency_ms_mean"]
            pwr   = row["mean_cpu_power_w_mean"] + (row["mean_gpu_power_w_mean"] if row["device"] == "cuda" else 0)
            y     = (lat / 1000) * pwr / 3600 * 1e6
            # error propagation for product: σ_E/E = sqrt((σ_L/L)² + (σ_P/P)²)
            s_lat = row.get("mean_latency_ms_std", 0) or 0
            s_pwr = (((row.get("mean_cpu_power_w_std", 0) or 0) ** 2 + (row.get("mean_gpu_power_w_std", 0) or 0) ** 2) ** 0.5
                        if row["device"] == "cuda" else (row.get("mean_cpu_power_w_std", 0) or 0))
            rel   = ((s_lat / lat) ** 2 + (s_pwr / pwr) ** 2) ** 0.5 if lat and pwr else 0
            yerr  = y * rel

            ax.errorbar(x, y, yerr=yerr, fmt="none", color=color,
                        alpha=0.6, capsize=4, linewidth=1.2)
            ax.scatter(x, y, color=color, marker=marker, s=size,
                       zorder=5, edgecolors="white", linewidths=0.5)

        ax.set_xticks(list(range(len(methods_present))))
        ax.set_xticklabels(
            [METHOD_LABELS.get(m, m) for m in methods_present],
            fontsize=8, rotation=20, ha="right"
        )
        ax.set_title(DATASET_LABELS.get(dataset, dataset), fontsize=10, fontweight="bold")
        ax.set_ylabel("Energy (µWh / image)" if ax == axes[0] else "")

    fig.legend(handles=_legend_handles(), loc="lower center", ncol=4,
               bbox_to_anchor=(0.5, -0.12), frameon=False, fontsize=9)
    plt.tight_layout()
    out = output_dir / "fig3_energy_by_method.png"
    fig.savefig(out, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")


# ---------------------------------------------------------------------------
# Figure 4: CPU vs GPU speedup ratio (latency_cpu / latency_gpu)
# ---------------------------------------------------------------------------

def plot_speedup(df: pd.DataFrame, output_dir: Path):
    cpu = df[df["device"] == "cpu"][
        ["dataset", "model_name", "pruning_method", "stored_precision",
         "mean_latency_ms_mean"]
    ].rename(columns={"mean_latency_ms_mean": "lat_cpu"})

    gpu = df[df["device"] == "cuda"][
        ["dataset", "model_name", "pruning_method", "stored_precision",
         "mean_latency_ms_mean"]
    ].rename(columns={"mean_latency_ms_mean": "lat_gpu"})

    merged = cpu.merge(gpu, on=["dataset", "model_name", "pruning_method", "stored_precision"])
    # Slowdown: how many times slower is CPU vs GPU
    merged["slowdown"] = merged["lat_cpu"] / merged["lat_gpu"]

    fig, axes = plt.subplots(1, len(DATASETS), figsize=(18, 5), sharey=True)
    fig.suptitle("CPU Slowdown Factor Relative to GPU (lat_cpu / lat_gpu)",
                 fontsize=14, fontweight="bold", y=1.02)

    for ax, dataset in zip(axes, DATASETS):
        sub = merged[merged["dataset"] == dataset].copy()
        if sub.empty:
            ax.set_visible(False)
            continue

        methods_present = [m for m in METHOD_ORDER if m in sub["pruning_method"].values]
        x_pos = _x_positions(methods_present, METHOD_ORDER)

        for _, row in sub.iterrows():
            m = row["pruning_method"]
            if m not in x_pos:
                continue
            x = x_pos[m]
            precision = row["stored_precision"]
            marker = PRECISION_MARKERS.get(precision, "s")
            color = "#1565c0" if precision == "fp32" else "#e65100"

            ax.scatter(x, row["slowdown"], color=color, marker=marker,
                       s=90, zorder=5, edgecolors="white", linewidths=0.5)

        # Reference line at y=1 (no slowdown — CPU == GPU)
        ax.axhline(1, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

        ax.set_xticks(list(range(len(methods_present))))
        ax.set_xticklabels(
            [METHOD_LABELS.get(m, m) for m in methods_present],
            fontsize=8, rotation=20, ha="right"
        )
        ax.set_title(DATASET_LABELS.get(dataset, dataset), fontsize=10, fontweight="bold")
        ax.set_ylabel("Slowdown Factor (×)" if ax == axes[0] else "")

    handles = [
        mlines.Line2D([], [], color="#1565c0", marker="^", linestyle="None",
                      markersize=8, label="FP32"),
        mlines.Line2D([], [], color="#e65100", marker="o", linestyle="None",
                      markersize=8, label="FP16"),
        mlines.Line2D([], [], color="gray", linestyle="--", linewidth=1,
                      label="No slowdown (×1)"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=3,
               bbox_to_anchor=(0.5, -0.12), frameon=False, fontsize=9)
    plt.tight_layout()
    out = output_dir / "fig4_cpu_slowdown.png"
    fig.savefig(out, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")


# ---------------------------------------------------------------------------
# Figure 5: Model size (MB) vs mean latency — scatter with method colour
# ---------------------------------------------------------------------------

def plot_size_vs_latency(df: pd.DataFrame, output_dir: Path):
    method_colors = {
        "baseline":         "#455a64",
        "regional_pruning": "#1565c0",
        "hybrid_pruning":   "#6a1b9a",
        "slim_kd":          "#2e7d32",
    }

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Model Size vs. Inference Latency",
                 fontsize=14, fontweight="bold")

    for ax, device in zip(axes, ["cpu", "cuda"]):
        sub = df[df["device"] == device].copy()
        for _, row in sub.iterrows():
            m = row["pruning_method"]
            color = method_colors.get(m, "gray")
            marker = PRECISION_MARKERS.get(row["stored_precision"], "s")
            ax.scatter(
                row["model_size_mb"],
                row["mean_latency_ms_mean"],
                color=color, marker=marker, s=80,
                edgecolors="white", linewidths=0.5, zorder=5,
            )

        ax.set_xlabel("Model Size (MB)")
        ax.set_ylabel("Mean Latency (ms)")
        ax.set_title(f"Device: {device.upper()}", fontweight="bold")

    # Legend: methods
    method_handles = [
        mpatches.Patch(color=c, label=METHOD_LABELS.get(m, m))
        for m, c in method_colors.items()
    ]
    prec_handles = [
        mlines.Line2D([], [], color="gray", marker="^", linestyle="None",
                      markersize=8, label="FP32"),
        mlines.Line2D([], [], color="gray", marker="o", linestyle="None",
                      markersize=8, label="FP16"),
    ]
    fig.legend(handles=method_handles + prec_handles,
               loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.14),
               frameon=False, fontsize=9)
    plt.tight_layout()
    out = output_dir / "fig5_size_vs_latency.png"
    fig.savefig(out, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")


# ---------------------------------------------------------------------------
# Table 1: Summary — mean latency + throughput + energy, averaged over datasets
# ---------------------------------------------------------------------------

def make_summary_table(df: pd.DataFrame, output_dir: Path):
    keep = ["pruning_method", "stored_precision", "device",
            "mean_latency_ms_mean", "throughput_imgs_per_s_mean",
            "mean_cpu_power_w_mean", "mean_gpu_power_w_mean", "model_size_mb", "num_params"]

    sub = df[keep].copy()
    sub["_power"] = np.where(sub["device"] == "cuda",
        sub["mean_cpu_power_w_mean"] + sub["mean_gpu_power_w_mean"],
        sub["mean_cpu_power_w_mean"])
    sub["mean_energy_uwh"] = (sub["mean_latency_ms_mean"] / 1000) * sub["_power"] / 3600 * 1e6
    sub["model_size_mb"] = sub["model_size_mb"].round(1)
    sub["num_params_M"] = (sub["num_params"] / 1e6).round(2)

    agg = (
        sub.groupby(["pruning_method", "stored_precision", "device"])
        .agg(
            latency_ms=("mean_latency_ms_mean", "mean"),
            throughput=("throughput_imgs_per_s_mean", "mean"),
            energy_uwh=("mean_energy_uwh", "mean"),
            model_size_mb=("model_size_mb", "first"),
            num_params_M=("num_params_M", "first"),
        )
        .reset_index()
    )

    agg["method_display"] = agg["pruning_method"].map(METHOD_LABELS).fillna(agg["pruning_method"])
    agg = agg.sort_values(["pruning_method", "stored_precision", "device"])

    # Round for display
    agg["latency_ms"] = agg["latency_ms"].round(2)
    agg["throughput"] = agg["throughput"].round(1)
    agg["energy_uwh"] = agg["energy_uwh"].round(3)

    display_cols = {
        "method_display": "Method",
        "stored_precision": "Precision",
        "device": "Device",
        "num_params_M": "Params (M)",
        "model_size_mb": "Size (MB)",
        "latency_ms": "Latency (ms)",
        "throughput": "Throughput (img/s)",
        "energy_uwh": "Energy (µWh/img)",
    }

    out_df = agg[list(display_cols.keys())].rename(columns=display_cols)
    out_path = output_dir / "table1_summary.csv"
    out_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

    # Also render as a matplotlib table figure
    fig, ax = plt.subplots(figsize=(16, 0.4 * len(out_df) + 1.5))
    ax.axis("off")
    tbl = ax.table(
        cellText=out_df.values,
        colLabels=out_df.columns,
        cellLoc="center",
        loc="center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.auto_set_column_width(col=list(range(len(out_df.columns))))

    # Header styling
    for j in range(len(out_df.columns)):
        tbl[(0, j)].set_facecolor("#37474f")
        tbl[(0, j)].set_text_props(color="white", fontweight="bold")

    # Row banding
    for i in range(1, len(out_df) + 1):
        color = "#f5f5f5" if i % 2 == 0 else "white"
        for j in range(len(out_df.columns)):
            tbl[(i, j)].set_facecolor(color)

    fig.suptitle("CNN Single-Image Benchmark Summary (averaged over 4 datasets)",
                 fontsize=10, fontweight="bold", y=0.98)
    plt.tight_layout()
    out_fig = output_dir / "table1_summary.png"
    fig.savefig(out_fig, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_fig}")

    return out_df


# ---------------------------------------------------------------------------
# Table 2: Per-dataset breakdown
# ---------------------------------------------------------------------------

def make_per_dataset_table(df: pd.DataFrame, output_dir: Path):
    keep = ["dataset", "pruning_method", "stored_precision", "device",
            "mean_latency_ms_mean", "throughput_imgs_per_s_mean",
            "mean_cpu_power_w_mean", "mean_gpu_power_w_mean"]

    sub = df[keep].copy()
    sub["_power"] = np.where(sub["device"] == "cuda",
        sub["mean_cpu_power_w_mean"] + sub["mean_gpu_power_w_mean"],
        sub["mean_cpu_power_w_mean"])
    sub["energy_uwh"] = ((sub["mean_latency_ms_mean"] / 1000) * sub["_power"] / 3600 * 1e6).round(3)
    sub["latency_ms"] = sub["mean_latency_ms_mean"].round(2)
    sub["throughput"] = sub["throughput_imgs_per_s_mean"].round(1)
    sub["method_display"] = sub["pruning_method"].map(METHOD_LABELS).fillna(sub["pruning_method"])

    out_df = sub[["dataset", "method_display", "stored_precision", "device",
                  "latency_ms", "throughput", "energy_uwh"]].rename(columns={
        "dataset": "Dataset",
        "method_display": "Method",
        "stored_precision": "Precision",
        "device": "Device",
        "latency_ms": "Latency (ms)",
        "throughput": "Throughput (img/s)",
        "energy_uwh": "Energy (µWh/img)",
    })
    out_df = out_df.sort_values(["Dataset", "Method", "Precision", "Device"])

    out_path = output_dir / "table2_per_dataset.csv"
    out_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")
    return out_df


# ---------------------------------------------------------------------------
# Paths — edit these two lines, then run the cell
# ---------------------------------------------------------------------------

INPUT_CSV  = Path("/Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/CNN/CNN_averaged.csv")   # ← change this
OUTPUT_DIR = Path("/Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/CNN/")          # ← change this


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main(input_csv: Path, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Loading: {input_csv}")
    df = load_data(input_csv)
    print(f"Rows after loading: {len(df)}")
    print(f"Methods:    {sorted(df['pruning_method'].unique())}")
    print(f"Devices:    {sorted(df['device'].unique())}")
    print(f"Precisions: {sorted(df['stored_precision'].unique())}")
    print()

    plot_latency(df, output_dir)
    plot_throughput(df, output_dir)
    plot_energy(df, output_dir)
    plot_speedup(df, output_dir)
    plot_size_vs_latency(df, output_dir)
    make_summary_table(df, output_dir)
    make_per_dataset_table(df, output_dir)

    print("\nAll outputs written to:", output_dir)


main(INPUT_CSV, OUTPUT_DIR)

Loading: /Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/CNN/CNN_averaged.csv
Rows after loading: 64
Methods:    ['baseline', 'hybrid_pruning', 'regional_pruning', 'slim_kd']
Devices:    ['cpu', 'cuda']
Precisions: ['fp16', 'fp32']

Saved: /Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/CNN/fig1_latency_by_method.png
Saved: /Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/CNN/fig2_throughput_by_method.png
Saved: /Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/CNN/fig3_energy_by_method.png
Saved: /Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/CNN/fig4_cpu_slowdown.png
Saved: /Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/CNN/fig5_size_vs_latency.png
Saved: /Users/arihangupta/Downloads/pruning_project_data/PruneAndTrain/SingleImageCPUGPU/CNN/table1_summary.csv
Saved: /Users/arihangupta/